# Experiment: BAT (behaviour-aware)

Re-implementation of BAT (Liao et al., AAAI 2024; [code](https://github.com/Petrichor625/BATraj-Behavior-aware-Model)). Four parts: a behaviour-aware module that turns graph centrality of every vehicle (degree, closeness, eigenvector) and its change per step into behaviour features; interaction-aware encoding of neighbours in polar coordinates; priority-aware attention over neighbours; and a manoeuvre-conditioned decoder in which each (lateral, longitudinal) manoeuvre learns its own mapping from history steps to future steps. Paper NGSIM RMSE: 0.23 / 0.81 / 1.54 / 2.52 / 3.62 m.

Part of the model comparison in `experiments/MODEL_ALTERNATIVES.md`. Run All; set `DEBUG = False` in section 2 for the real run.

## 0. Decisions & assumptions

Every choice that could change the result is listed here; the cell notes below repeat the ones that apply to each cell.

**Shared by every notebook (S)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| S1 | Same data for every model: NGSIM us-101 + i-80 (DEBUG: first 250k rows of us-101), scene-level 70/10/20 split, seed 42, train-only normalization. | Differences in the results must come from the model, not the data. | Papers use their own splits, so published numbers are not like-for-like with ours. |
| S2 | Same training budget for every model: config.py defaults (12 epochs, Adam lr 5e-4 decayed x0.6 per epoch, batch 128, weight decay 1e-5, grad clip 1.0). | A fair comparison; per-model tuning would favour whichever model got tuned most. | Each paper's own schedule (e.g. CS-LSTM trains longer with lr 1e-3). Tune later with --mode tune if a model looks under-trained. |
| S3 | Loss = MSE + bivariate-Gaussian NLL + neighbour-goal MSE (`basic_loss`); the risk-scaled loss is used only where this notebook's M table says so (STRAP-R-based variants). Models without a goal head output zero goals, so their goal term is a constant. A model may add its own `aux_loss` (only the diffusion model does). | Keeps one training objective; the risk-scaled loss is STRAP-R's own contribution. | Loss values are therefore not comparable across models: compare RMSE only. |
| S4 | Best checkpoint = lowest validation loss (`fit` default); test RMSE is reported at 1-5 s for that checkpoint. | Same selection rule for every model. | Selecting on val RMSE would be closer to the reported metric. |
| S5 | KNOWN LIMITATION (not fixed): positions are relative to the target at each timestep, so the target's own history is always (0, 0) and lateral speed is always 0 (NGSIM gives longitudinal speed only). The target is seen through speed, acceleration and lane features; lateral context comes from neighbours. | Kept as is (user decision) so every model gets identical inputs. | Papers feed the target's own x/y track; lane-change prediction is handicapped for all models here. |
| S6 | Neighbours = up to 15 risk-selected vehicles (S/O-field > RISK_THRESHOLD), not every nearby vehicle. | Same scene construction as STRAP. | Grid/graph papers use all vehicles in range; ours see only the risk-selected ones. |
| S7 | Output = one bivariate Gaussian per future step (mu_x, mu_y, sigma_x, sigma_y, rho); RMSE uses the mean. Multimodal models are moment-matched to one Gaussian. | The `fit` / `evaluate_trajectory` contract and how STRAP's Table I is scored. | minADE/minFDE over modes would favour multimodal models. |

**Specific to this model (M)**

| # | Decision / assumption | Why | Departure from paper / alternative |
|---|---|---|---|
| M1 | Structure follows the public code (`model5f_BAT_new.py`); sizes rounded to 64 (traj embedding 32, encoders 64, behaviour embedding 4 per vehicle, 4 heads). | Public code read for module layout; the repo itself says it is a reconstruction of the paper. | BAT uses a 16-unit trajectory LSTM and 48-unit attention outputs. |
| M2 | Behaviour graph per history step: all real vehicles, edge weight exp(-d / 10 m), no self-loops; degree = mean edge weight, closeness = (n-1) / sum of distances (x10 m), eigenvector = 20 power iterations; plus each one's step-to-step change (6 features). | BAT's levels + differences of degree, closeness and eigenvector centrality. | ASSUMPTION: the 10 m scale is ours. BAT uses exp(-d) on polar (r, phi) pairs, and its code has bugs (degree/eigenvector levels copy closeness, their differences are always 0); we compute what the paper describes. |
| M3 | Polar coordinates (r / 30 m, phi) of every vehicle relative to the target are appended to the 11 features (13 inputs). | BAT encodes neighbours in polar coordinates. | The target's own polar track is (0, 0) here (S5). |
| M4 | Priority-aware attention: self-attention among real vehicles at the last step, then a masked mean; broadcast over time and fused per step with the target's own LSTM sequence and the behaviour sequence through a GLU. | Matches the code's attention + pooling + GLU fusion. | BAT pools over a 13 x 3 grid occupancy; we pool over the risk-selected neighbours (S6). |
| M5 | 9 LATENT manoeuvres (3 lateral x 3 longitudinal); each gets a softmax mapping from 30 history steps to 50 future steps and a one-hot-conditioned LSTM decoder; modes weighted by p_lat x p_lon and moment-matched. | No manoeuvre labels reach the model (`fit` gives only states). | BAT trains the manoeuvre heads on labels and outputs the most likely manoeuvre. |
| M6 | Behaviour features use positions at every step; a neighbour missing at some step reads as the normalisation mean there. | Same inputs as all models. | Per-step masks would be cleaner; the loader only gives a per-vehicle mask. |

## 1. Setup

**Decisions in this cell:** seed fixed to `SEED` (42) in every notebook so initialisation is not a source of differences (S1).

In [ ]:
import sys
from pathlib import Path

# Repo root = first folder upwards that contains src/ (works from experiments/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.config import (
    BATCH_SIZE, BETA_LOSS, DEBUG_MAX_ROWS, DEVICE, DROPOUT, DT, LEARNING_RATE, LR_DECAY, N_V,
    NGSIM_LOCATIONS, NUM_EPOCHS, NORMALIZATION_FILE, SEED, STATE_DIM, TF_STEPS, TH_STEPS, WEIGHT_DECAY,
)
from src.data.loader import NGSIMDataLoader, create_dataloader
from src.data.normalization import load_stats, normalize_samples
from src.evaluation.evaluate import evaluate_trajectory
from src.models.strap import STRAP
from src.pipeline import collision_scores, metric_rows, test_groups
from src.train import fit, prepare_data
from experiments.benchmark import compare, save_result

torch.manual_seed(SEED)
print("device:", DEVICE)

## 2. Experiment config

**Decisions in this cell:** training budget is config.py's (S2); `use_risk_loss` as in the M table; collisions scored at 2 m like `--mode collisions`.

In [ ]:
EXP_NAME = "bat"          # unique per notebook -> experiments/results/<EXP_NAME>.json
DESCRIPTION = "BAT-style: centrality behaviour features + polar pooling + priority attention, 9 latent manoeuvres"

DEBUG = True                  # True: DEBUG_MAX_ROWS of us-101 for a quick check; False: full data
LOCATIONS = ["us-101"] if DEBUG else NGSIM_LOCATIONS

HPARAMS = dict(               # keyword args of src.train.fit
    num_epochs=NUM_EPOCHS,
    use_risk_loss=False,       # False = basic loss (STRAP-B style)
    beta=BETA_LOSS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    lr_decay=LR_DECAY,
    weight_decay=WEIGHT_DECAY,
)

RUN_COLLISIONS = True         # per-pair collision scoring on the test split (slow on full data)
COLLISION_THRESHOLD = 2.0     # metres, GT collision label distance

CKPT_DIR = ROOT / "experiments" / "checkpoints" / EXP_NAME

## 3. Data
Train-only normalization stats and the intention codebook are written to `CKPT_DIR`.

**Decisions in this cell:** same loader, split and train-only normalization as every other notebook (S1, S6); `norm_stats` is loaded here so the model (next section) can recover metres and lane numbers.

In [ ]:
loader = NGSIMDataLoader(locations=LOCATIONS, max_rows=DEBUG_MAX_ROWS if DEBUG else None)
train_s, val_s, centers = prepare_data(loader, save_dir=str(CKPT_DIR))
_, _, test_raw = loader.get_splits()   # cached; raw (unnormalized) test split
# Train-only mean/std; models that need metres or lane numbers undo the normalization with it.
norm_stats = load_stats(str(CKPT_DIR / NORMALIZATION_FILE))
print(f"train/val/test = {len(train_s)}/{len(val_s)}/{len(test_raw)}")

## 4. Model

Any `torch.nn.Module` works if `forward(states, mask, dims)` returns a 3-tuple:

| output | shape | meaning |
|---|---|---|
| `traj_dist` | `(B, T_f=50, 5)` | `[mu_x, mu_y, sigma_x, sigma_y, rho]` of the target, metres relative to its last observed position (x longitudinal, y lateral); `sigma > 0`, `rho` in (-1, 1) |
| `goals` | `(B, N-1, 4)` | neighbour goals; a model without a goal head returns `states.new_zeros(B, N - 1, 4)` |
| third | anything | ignored by training/eval (STRAP returns its risk field) |

Inputs: `states (B, T_h=30, N, 11)` normalized (vehicle 0 = target, rest neighbours, zero-padded),
`mask (B, N)` True = real vehicle, `dims (B, N, 2)` raw length/width in metres.
Feature index: 0 rel-longitudinal m, 1 rel-lateral m, 2 speed/30, 3 lateral speed (always 0), 4 accel/10,
5 type/2, 6 relative lane (target - vehicle, left = +), 7 length/10, 8 width/3, 9 S-field risk, 10 O-field risk.
Raw value = `states * std + mean` with `norm_stats`. STRAP's k-means codebook is only loaded if the model has a `risk_decoder`.

**Decisions in this cell:** layout from BAT's public code, sizes rounded (M1); behaviour centralities computed per step from denormalized positions (M2, M6); polar inputs (M3); priority attention + GLU fusion (M4); 9 latent manoeuvres with learned time mappings (M5). Built after the data cell because it needs `norm_stats`.

In [ ]:
def moment_match(raw, weights):
    """(B, M, T, 5) raw mode outputs + (B, M) mode weights -> (B, T, 5) one Gaussian (S7)."""
    mu = raw[..., :2]
    sig = F.softplus(raw[..., 2:4]) + 1e-3
    rho = torch.tanh(raw[..., 4])
    w = weights[:, :, None]                                         # (B, M, 1)
    mean = (w[..., None] * mu).sum(1)                               # (B, T, 2)
    d = mu - mean[:, None]
    var = (w[..., None] * (sig ** 2 + d ** 2)).sum(1)
    cov = (w * (rho * sig[..., 0] * sig[..., 1] + d[..., 0] * d[..., 1])).sum(1)
    sx, sy = var[..., 0].sqrt(), var[..., 1].sqrt()
    return torch.stack([mean[..., 0], mean[..., 1], sx, sy, (cov / (sx * sy)).clamp(-0.999, 0.999)], dim=-1)

def behaviour_features(pos, mask, scale=10.0, iters=20):
    """Per-step centralities (M2). pos (B, T, N, 2) metres, mask (B, N) -> (B, T, N, 6): levels + changes."""
    N = pos.shape[2]
    d = torch.cdist(pos, pos)                                       # (B, T, N, N)
    pair = (mask[:, None, :, None] & mask[:, None, None, :]).float() * (1 - torch.eye(N, device=pos.device))
    W = torch.exp(-d / scale) * pair
    n = mask.sum(-1).float()[:, None, None]                         # (B, 1, 1) real vehicles
    degree = W.sum(-1) / (n - 1).clamp(min=1)
    dist_sum = (d * pair).sum(-1)
    closeness = torch.where(dist_sum > 0, (n - 1) * scale / dist_sum.clamp(min=1e-6), torch.zeros_like(dist_sum))
    eig = mask[:, None].float().expand_as(degree).clone()
    for _ in range(iters):                                          # power iteration
        eig = (W @ eig[..., None]).squeeze(-1)
        eig = eig / eig.norm(dim=-1, keepdim=True).clamp(min=1e-6)
    level = torch.stack([degree, closeness, eig], dim=-1) * mask[:, None, :, None].float()
    change = torch.cat([torch.zeros_like(level[:, :1]), level[:, 1:] - level[:, :-1]], dim=1)
    return torch.cat([level, change], dim=-1)


class BAT(nn.Module):
    """Behaviour-aware trajectory prediction (BAT-style)."""
    def __init__(self, norm_stats, input_dim=STATE_DIM + 2, emb=32, d=64, beh=4, heads=4,
                 n_lat=3, n_lon=3, n_nodes=N_V + 1, T_h=TH_STEPS, T_f=TF_STEPS):
        super().__init__()
        self.register_buffer("mean", torch.tensor(norm_stats["mean"]))
        self.register_buffer("std", torch.tensor(norm_stats["std"]))
        self.n_nodes, self.T_f, self.n_lat, self.n_lon = n_nodes, T_f, n_lat, n_lon
        self.traj_embed = nn.Linear(input_dim, emb)
        self.traj_lstm = nn.LSTM(emb, d, batch_first=True)          # shared by target and neighbours
        self.beh_embed = nn.Linear(6, beh)
        self.beh_lstm = nn.LSTM(beh * n_nodes, d, batch_first=True)  # all vehicles' behaviour, flattened
        self.priority = nn.MultiheadAttention(d, heads, batch_first=True)
        self.fuse = nn.Linear(3 * d, 2 * d)                         # GLU fusion
        self.man_fc = nn.Sequential(nn.Linear(d, d), nn.LayerNorm(d))
        self.op_lat, self.op_lon = nn.Linear(d, n_lat), nn.Linear(d, n_lon)
        self.mapping = nn.Parameter(nn.init.xavier_uniform_(torch.empty(T_h, T_f, n_lat + n_lon)))
        lat, lon = torch.meshgrid(torch.arange(n_lat), torch.arange(n_lon), indexing="ij")
        self.register_buffer("onehot", torch.cat([F.one_hot(lat.flatten(), n_lat), F.one_hot(lon.flatten(), n_lon)], -1).float())
        self.dec_in = nn.Linear(d + n_lat + n_lon, d)
        self.decoder = nn.LSTM(d, d, batch_first=True)
        self.out = nn.Linear(d, 5)

    def forward(self, states, mask, dims=None):
        B, T_h, N, _ = states.shape
        pos = states[..., :2] * self.std[:2] + self.mean[:2]        # metres relative to the target
        r = pos.norm(dim=-1, keepdim=True)
        phi = torch.atan2(pos[..., 1:], pos[..., :1])
        x = torch.cat([states, r / 30.0, phi], dim=-1)              # M3: polar inputs
        h, _ = self.traj_lstm(F.elu(self.traj_embed(x)).transpose(1, 2).reshape(B * N, T_h, -1))
        h = h.reshape(B, N, T_h, -1)
        # Behaviour-aware (M2): every vehicle's centralities, padded to n_nodes and flattened per step.
        beh = F.leaky_relu(self.beh_embed(behaviour_features(pos, mask)), 0.1)
        beh = F.pad(beh, (0, 0, 0, self.n_nodes - N)).reshape(B, T_h, -1)
        beh, _ = self.beh_lstm(beh)
        # Priority-aware (M4): self-attention among real vehicles at the last step, masked mean.
        last = h[:, :, -1]
        att, _ = self.priority(last, last, last, key_padding_mask=~mask)
        m = mask[..., None].float()
        nbr = (att * m).sum(1) / m.sum(1)
        fused = F.glu(self.fuse(torch.cat([h[:, 0], nbr[:, None].expand(-1, T_h, -1), beh], -1)), dim=-1)  # (B, T_h, d)
        # Manoeuvres (M5): probabilities from the last step, one learned history->future mapping per (lat, lon).
        state = self.man_fc(fused[:, -1])
        p = (F.softmax(self.op_lat(state), -1)[:, :, None] * F.softmax(self.op_lon(state), -1)[:, None]).flatten(1)  # (B, 9)
        maps = F.softmax((self.mapping @ self.onehot.T).permute(2, 1, 0), dim=-1)    # (9, T_f, T_h)
        dec = torch.einsum("mfh,bhd->bmfd", maps, fused)                             # (B, 9, T_f, d)
        M = maps.shape[0]
        dec = torch.cat([dec, self.onehot[None, :, None].expand(B, -1, self.T_f, -1)], -1)
        out, _ = self.decoder(F.elu(self.dec_in(dec)).reshape(B * M, self.T_f, -1))
        raw = self.out(out).reshape(B, M, self.T_f, 5)
        return moment_match(raw, p), states.new_zeros(B, N - 1, 4), None


def build_model():
    return BAT(norm_stats)

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params:,} parameters")

## 5. Train

**Decisions in this cell:** `fit` with the config budget (S2); best checkpoint by val loss (S4). Loss curves are only comparable within this notebook (S3).

In [ ]:
t0 = time.time()
training_log = fit(model, train_s, val_s, centers, save_dir=str(CKPT_DIR), **HPARAMS)
train_minutes = (time.time() - t0) / 60

log_df = pd.DataFrame(training_log)
display(log_df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
log_df.plot(x="epoch", y=["train_loss", "val_loss"], marker=".", ax=axes[0], title="Loss")
log_df.plot(x="epoch", y="val_rmse", marker=".", ax=axes[1], title="Val RMSE (m)")
plt.tight_layout(); plt.show()

## 6. Evaluate (best checkpoint) — RMSE 1s…5s overall, per site, zero-neighbour scenes

**Decisions in this cell:** RMSE of the predicted mean at 1-5 s on the test split, overall / per site / target-only scenes (S7).

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / "best_model.pt", map_location=DEVICE, weights_only=True))
model.eval()
test_s = normalize_samples(test_raw, norm_stats)

test_metrics = {}
for label, samples in test_groups(test_s).items():
    dl = create_dataloader(samples, batch_size=32, shuffle=False)
    test_metrics[label] = {**evaluate_trajectory(model, dl, DEVICE), "n": len(samples)}
display(pd.DataFrame(test_metrics).T)

## 7. Collision prediction (optional)
Model's predicted target vs constant-velocity neighbours, every risk metric at its default threshold.

**Decisions in this cell:** neighbours are rolled forward at constant velocity and candidates are the nearest vehicles at the last history step, as in `--mode collisions`; only the target trajectory comes from this model.

In [ ]:
collisions = None
if RUN_COLLISIONS:
    scores, gt, _ = collision_scores(model, loader, test_raw, norm_stats, COLLISION_THRESHOLD)
    collisions = {"threshold": COLLISION_THRESHOLD, "n_pairs": len(gt),
                  "n_gt_collisions": int(gt.sum()), "rows": metric_rows(scores, gt)}
    print(f"{len(gt)} pairs, {int(gt.sum())} GT collisions")
    display(pd.DataFrame(collisions["rows"]).set_index("metric"))

## 8. Save result

**Decisions in this cell:** one JSON per `EXP_NAME`; a rerun overwrites it.

In [ ]:
save_result({
    "name": EXP_NAME,
    "description": DESCRIPTION,
    "model_class": type(model).__name__,
    "n_params": n_params,
    "hparams": HPARAMS,
    "data": {"locations": LOCATIONS, "debug": DEBUG},
    "train_minutes": train_minutes,
    "training_log": training_log,
    "test": test_metrics,
    "collisions": collisions,
})

## 9. Comparison & benchmarking — all results in `experiments/results/`

Ranked by overall test `rmse_avg` (best per column highlighted). Check the `data` column:
debug and full-data runs are listed together but are not comparable.
Also written to `experiments/comparison_rmse.csv` / `comparison_collisions.csv`.

**Decisions in this cell:** compares every saved result; only rows with the same `data` value are like-for-like.

In [ ]:
rmse_df, collision_df = compare()